# 02 Style language

Phase 2 has no maps yet: a style string compiles to a `StyleSpec`, and every mistake raises an error that points at the problem.

In [1]:
import kharita as kh

kh.help()

fill       Constant fill colour, or colour by data
stroke     Outline or line colour, or its width in pixels
size       Point radius in pixels
opacity    Whole-layer opacity
ramp       Colour ramp for data; -r reverses it
scheme     How data values are grouped into colour classes
null       Colour for missing values (default gray-300)
highlight  Colour of the feature under the mouse
basemap    Basemap theme
legend     Hide the legend or choose its corner
tooltip    Columns shown when hovering
extrude    3D height from data, and the height of the tallest feature
preset     Expand a built-in or kh.define() style

kh.help('fill') shows the syntax and examples for one utility.


In [2]:
# Details and examples for one utility
kh.help("scheme")

scheme: How data values are grouped into colour classes

Syntax:   scheme-q<k> | scheme-e<k> | scheme-[b1,b2] | scheme-cat | scheme-div
Examples: scheme-q5  scheme-e4  scheme-[10,50,100]  scheme-cat  scheme-div  scheme-div-[50]


In [3]:
# What a full style compiles to, and which token set each part
kh.explain("preset-choropleth fill-[pop/area_km2] ramp-ylorrd tooltip-[key,pop]")

fill          data: pop / area_km2      fill-[pop/area_km2]
ramp          ylorrd                    ramp-ylorrd
scheme        quantiles, 5 classes      scheme-q5 (from preset-choropleth)
null          rgba(209, 213, 219, 255)  null-gray-300 (from preset-choropleth)
stroke        rgba(255, 255, 255, 255)  stroke-white (from preset-choropleth)
stroke width  1 px                      stroke-1 (from preset-choropleth)
highlight     rgba(251, 191, 36, 255)   highlight-amber-400 (from preset-choropleth)
tooltip       key, pop                  tooltip-[key,pop]
basemap       light                     default
legend        bottom right              default
columns read  pop, area_km2, key

In [4]:
# The StyleSpec itself, which Phase 3 and 4 will turn into colours and a map
spec = kh.parse("fill-[density] ramp-ylorrd scheme-q5 stroke-1 stroke-white")
spec

StyleSpec(fill=None, fill_by=Expr(column='density', divisor=None, factor=None), ramp='ylorrd', ramp_reverse=False, scheme=Scheme(kind='quantiles', k=5, breaks=None, center=None), stroke=(255, 255, 255, 255), stroke_width=1.0, size=None, opacity=None, highlight=None, null=None, tooltip=None, extrude_by=None, extrude_max=None, basemap=None, legend=True, legend_position=None)

In [5]:
# Each of these fails with a message and a caret under the problem
bad = [
    "fil-red-500 stroke-1",
    "fill-indgo-600",
    "fill-red-500 fill-[pop]",
    "fill-[pop",
    "ramp-set2 scheme-q5 fill-[x]",
    "scheme-q12 fill-[x]",
]
for style in bad:
    try:
        kh.parse(style)
    except kh.StyleSyntaxError as err:
        print(err, end="\n\n")

Unknown utility 'fil-red-500'. Did you mean 'fill-red-500'? Run kh.help() to list every utility.
  fil-red-500 stroke-1
  ^^^^^^^^^^^

fill-indgo-600: Unknown colour family 'indgo'. Did you mean 'indigo'?
  fill-indgo-600
  ^^^^^^^^^^^^^^

'fill-[pop]' conflicts with 'fill-red-500'; both set fill.
  fill-red-500 fill-[pop]
               ^^^^^^^^^^

Bracket is never closed.
  fill-[pop
       ^

ramp-set2 is for categories; use scheme-cat or a sequential ramp.
  ramp-set2 scheme-q5 fill-[x]
  ^^^^^^^^^

scheme-q12: Use 3 to 9 classes; readers cannot match more than about 7, got 12.
  scheme-q12 fill-[x]
  ^^^^^^^^^^



In [6]:
# Check a style against the Sudan columns without drawing anything
import geopandas as gpd

states = gpd.read_file("../data/processed/sudan_states.gpkg")
kh.validate("fill-[pop/area_km2] tooltip-[key,pop]", columns=states.columns)
try:
    kh.validate("fill-[Density]", columns=states.columns)
except kh.BindingError as err:
    print(err)

Column 'Density' is not in the data. Did you mean 'density'? Column names are case-sensitive.


In [7]:
# Your own reusable style, then used as a preset
kh.define("sudan", "preset-choropleth ramp-ylorrd basemap-light legend-br")
kh.explain("preset-sudan fill-[density] scheme-e4")

fill          data: density              fill-[density]
ramp          ylorrd                     ramp-ylorrd (from preset-sudan)
scheme        equal interval, 4 classes  scheme-e4
null          rgba(209, 213, 219, 255)   null-gray-300 (from preset-choropleth)
stroke        rgba(255, 255, 255, 255)   stroke-white (from preset-choropleth)
stroke width  1 px                       stroke-1 (from preset-choropleth)
highlight     rgba(251, 191, 36, 255)    highlight-amber-400 (from preset-choropleth)
basemap       light                      basemap-light (from preset-sudan)
legend        bottom right               legend-br (from preset-sudan)
columns read  density